<a href="https://colab.research.google.com/github/tarannump096-cpu/ANN/blob/main/Simple_Recommendation_System_using_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy scikit-learn tensorflow

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten
from tensorflow.keras.layers import Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam

In [3]:
df = pd.read_csv("/content/ratings.csv")

In [4]:
print(df.head())

print(df.shape)

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
(100836, 4)


In [6]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user'] = user_encoder.fit_transform(df['userId'])
df['item'] = item_encoder.fit_transform(df['movieId'])

In [7]:
X = df[['user', 'item']]

y = df['rating']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
num_users = df['user'].nunique()

num_items = df['item'].nunique()

In [10]:
user_input = Input(shape=(1,))
item_input = Input(shape=(1,))

In [11]:
user_embedding = Embedding(
    input_dim=num_users,
    output_dim=50
)(user_input)

item_embedding = Embedding(
    input_dim=num_items,
    output_dim=50
)(item_input)

In [12]:
user_vec = Flatten()(user_embedding)

item_vec = Flatten()(item_embedding)

In [13]:
concat = Concatenate()([user_vec, item_vec])

In [14]:
dense1 = Dense(128, activation='relu')(concat)

drop1 = Dropout(0.3)(dense1)

dense2 = Dense(64, activation='relu')(drop1)

drop2 = Dropout(0.3)(dense2)

dense3 = Dense(32, activation='relu')(drop2)

In [15]:
output = Dense(1)(dense3)

In [16]:
model = Model(
    inputs=[user_input, item_input],
    outputs=output
)

In [17]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [18]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │     30,500 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │    486,200 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,928 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 539,997 (2.06 MB)

 Trainable params: 539,997 (2.06 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = model.fit(
    [X_train['user'], X_train['item']],
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - loss: 1.1793 - mae: 0.8225 - val_loss: 0.8118 - val_mae: 0.7061
Epoch 2/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - loss: 0.7588 - mae: 0.6761 - val_loss: 0.7783 - val_mae: 0.6808
Epoch 3/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.6848 - mae: 0.6376 - val_loss: 0.8124 - val_mae: 0.6812
Epoch 4/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - loss: 0.6391 - mae: 0.6116 - val_loss: 0.7732 - val_mae: 0.6757
Epoch 5/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - loss: 0.5939 - mae: 0.5872 - val_loss: 0.7738 - val_mae: 0.6778
Epoch 6/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - loss: 0.5472 - mae: 0.5610 - val_loss: 0.7871 - val_mae: 0.6809
Epoch 7/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.5027 - mae: 0.5357 - val_loss: 0.8287 - val_mae: 0.6934
Epoch 8/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.4655 - mae: 0.5142 - val_loss: 0.8199 - val_mae: 0.6894
Epoch 9/20
2017/2017 ━━━━━━━━━━━

In [20]:
loss, mae = model.evaluate([X_test['user'], X_test['item']], y_test)

print("Mean Absolute Error:", mae)

631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.8505 - mae: 0.6996
Mean Absolute Error: 0.6996282339096069


In [21]:
predictions = model.predict([X_test['user'], X_test['item']])

print(predictions[:10])

631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
[[3.3940892]
 [3.4706182]
 [2.9523516]
 [2.9203782]
 [3.6251192]
 [3.4310207]
 [3.9463687]
 [4.0217333]
 [2.54666  ]
 [4.2595005]]


In [23]:
user_id = 0

all_items = np.arange(num_items)

# Reshape user_array and all_items to (batch_size, 1)
user_array = np.array([user_id] * num_items).reshape(-1, 1)
item_array = all_items.reshape(-1, 1)

predicted_ratings = model.predict(
    [user_array, item_array]
)

top_items = np.argsort(predicted_ratings.flatten())[-5:]

print("Recommended Items:")

for item_encoded_id in top_items:
    original_item_id = item_encoder.inverse_transform([item_encoded_id])[0]
    print(original_item_id)

304/304 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Recommended Items:
147326
149566
2035
26587
147330


In [24]:
model.save("recommendation_ann_model.h5")